In [ ]:
from typing import TypedDict

from IPython.display import display, Image
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END

In [ ]:
load_dotenv(override=True)
llm = init_chat_model(
     model="nvidia/nemotron-3-nano-30b-a3b",
     model_provider="nvidia",
     temperature=0.0)

In [ ]:
class UserState(TypedDict):
    raw_text: str
    f_text: str
    word_count: int
    f_summary: str


def format_txt_node(state: UserState):
    f_text = state["raw_text"]
    res = llm.invoke(f"Generate content less than 50 words on the {f_text}")
    return {
        "f_text": res.content
    }


def word_count_node(state: UserState):
    w_count = len(state["f_text"].split())
    return {
        "word_count": w_count
    }


def summary_node(state: UserState):
    f_txt = state["f_text"]
    preview_text = f_txt[:-25] + "..." if len(f_txt) > 10 else ""
    quick_summary = f'''

     **** Summary ****
     Preview Txt: {preview_text}
     Word Count: {state['word_count']}
     *****************
      
    '''
    return {
        "f_summary": quick_summary
    }

In [ ]:
wf_builder = StateGraph(UserState)
wf_builder.add_node("format_node", format_txt_node)
wf_builder.add_node("word_count_node", word_count_node)
wf_builder.add_node("summary_node", summary_node)

wf_builder.add_edge(START, "format_node")
wf_builder.add_edge("format_node", "word_count_node")
wf_builder.add_edge("word_count_node", "summary_node")
wf_builder.add_edge("summary_node", END)

graph = wf_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
while True:
    _title = str(input("Topic Pls (/bye to exit): "))
    if _title == "/bye":
        break
    resp = graph.invoke({
        "raw_text": _title,
    })
    print(resp["f_summary"])